# Radar de Concursos de TI — Administração no Google Colab

Este notebook já fica dentro do GitHub. Abra pelo botão do README, execute as células, altere os dados e use **Exportar ZIP**. Não é necessário instalar Git na sua máquina.

In [ ]:
#@title 1. Baixar o projeto do GitHub
import os, json, shutil, subprocess, pathlib, re, unicodedata
REPOSITORIO = "https://github.com/osmarrcs/radar-concursos-ti.git" #@param {type:"string"}
PASTA = pathlib.Path("/content/radar-concursos-ti")
if PASTA.exists(): shutil.rmtree(PASTA)
subprocess.run(["git","clone",REPOSITORIO,str(PASTA)],check=True)
os.chdir(PASTA)
print("Projeto carregado em", PASTA)

In [ ]:
#@title 2. Carregar base
DATA_FILE=PASTA/'data/competitions.json'
payload=json.loads(DATA_FILE.read_text(encoding='utf-8'))
registros=payload.setdefault('competitions',[])
orgaos=sorted({(r['organ_id'],r['organ_acronym'],r['organ_name']) for r in registros},key=lambda x:x[1])
print(f"{len(orgaos)} órgãos, {len({r['contest_id'] for r in registros})} concursos e {len(registros)} cargos.")
for oid,sigla,nome in orgaos: print(f"- {sigla}: {nome}")

## Novo órgão
Para simplificar, informe apenas **nome completo e sigla**. O identificador é criado automaticamente. Esfera, carreira e escopo são inferidos por palavras conhecidas e podem ser corrigidos diretamente no registro do concurso quando necessário.

In [ ]:
#@title 3. Criar órgão simplificado
NOME_ORGAO = "" #@param {type:"string"}
SIGLA = "" #@param {type:"string"}

def slug(texto):
    texto=unicodedata.normalize('NFD',texto).encode('ascii','ignore').decode().lower()
    return re.sub(r'[^a-z0-9]+','-',texto).strip('-')

def inferir(nome,sigla):
    t=(nome+' '+sigla).lower()
    if any(x in t for x in ['instituto federal','universidade federal']): return 'Universidades e Institutos Federais','Federal','regional_federal'
    if any(x in t for x in ['tribunal','trt','trf','tre','tj']): return 'Tribunais','Federal' if any(x in t for x in ['trt','trf','tre']) else 'Estadual','regional_federal' if any(x in t for x in ['trt','trf','tre']) else 'regional'
    if any(x in t for x in ['dataprev','serpro','banco central','caixa','banco do brasil']): return 'Empresas e Órgãos Federais','Federal','national'
    if any(x in t for x in ['prefeitura','emprel']): return 'Prefeituras e Empresas Municipais','Municipal','regional'
    return 'Outros órgãos','Não inferida','regional'

if NOME_ORGAO.strip() and SIGLA.strip():
    carreira,esfera,escopo=inferir(NOME_ORGAO,SIGLA)
    novo_orgao={'organ_id':slug(SIGLA),'organ_name':NOME_ORGAO.strip(),'organ_acronym':SIGLA.strip().upper(),'career':carreira,'sphere':esfera,'scope':escopo}
    print(novo_orgao)
else:
    novo_orgao=None
    print('Preencha nome e sigla somente quando precisar adicionar um órgão novo.')

## Cadastrar cargo dentro de um concurso
Um concurso pode ter várias linhas, uma por cargo/especialidade. Assim, o portal mostra o edital e depois todas as opções de TI: segurança, infraestrutura, dados, governança, redes, suporte, sistemas e laboratórios.

In [ ]:
#@title 4. Adicionar ou atualizar cargo/especialidade
ORGAO_ID = "ifpe" #@param {type:"string"}
CONCURSO_ID = "ifpe-tae-2025" #@param {type:"string"}
TITULO_CONCURSO = "Concurso TAE IFPE 2025" #@param {type:"string"}
ANO = 2025 #@param {type:"integer"}
CATEGORIA_TI = "Infraestrutura e Segurança" #@param ["Segurança da Informação","Governança de TI","Dados e Banco de Dados","Infraestrutura e Segurança","Redes e Infraestrutura","Suporte Técnico","Desenvolvimento e Sistemas","Laboratórios de TI","Laboratórios e Redes","Laboratórios e Suporte","Tecnologia da Informação"]
CARGO = "Analista de Tecnologia da Informação" #@param {type:"string"}
ESPECIALIDADE = "Infraestrutura e Segurança" #@param {type:"string"}
VAGAS = 1 #@param {type:"integer"}
VALIDADE = "2027-12-09" #@param {type:"string"}
LINK_EDITAL = "" #@param {type:"string"}
LINK_RESULTADO = "" #@param {type:"string"}
LINK_NOMEACAO = "" #@param {type:"string"}

base=next((r for r in registros if r['organ_id']==ORGAO_ID),None)
if not base and novo_orgao and novo_orgao['organ_id']==ORGAO_ID: base=novo_orgao
if not base: raise ValueError('Órgão não localizado. Cadastre nome e sigla na célula anterior ou use um ID existente.')
rid=slug(f'{CONCURSO_ID}-{CARGO}-{ESPECIALIDADE}')
fontes=[{'label':a,'url':u} for a,u in [('Edital',LINK_EDITAL),('Resultado final',LINK_RESULTADO),('Nomeação/convocação',LINK_NOMEACAO)] if u]
registro={
'id':rid,'organ_id':ORGAO_ID,'organ_name':base['organ_name'],'organ_acronym':base['organ_acronym'],'career':base.get('career','Outros órgãos'),'sphere':base.get('sphere','Não inferida'),'scope':base.get('scope','regional'),'state':base.get('state','PE' if ORGAO_ID=='ifpe' else 'BR'),'city':base.get('city',''),
'contest_id':CONCURSO_ID,'contest_title':TITULO_CONCURSO,'year':ANO,'it_category':CATEGORIA_TI,'position':CARGO,'specialty':ESPECIALIDADE,'status':'Cadastrado','immediate_vacancies':VAGAS,'reserve_list':None,'valid_until':VALIDADE,'last_called_rank':None,'last_called_score':None,'total_appointed':None,'quota_type':'Ampla concorrência','lotation':'Não informada','confidence':'baixa','is_official':bool(LINK_EDITAL),'verified_at':'2026-07-31','sources':fontes,
'collection_status':{'found':bool(LINK_EDITAL),'code':'OK' if LINK_EDITAL else 'MISSING_OFFICIAL_SOURCE','reason':'Fonte vinculada.' if LINK_EDITAL else 'Inclua o link oficial para confirmar o registro.','source':'cadastro_colab'},
'vacancy':{'count':None,'reference_date':'','reason':'Vacância ainda não apurada.','sources':[],'collection_status':{'found':False,'code':'VACANCY_NOT_FOUND','reason':'Use a célula específica de vacância.','source':'vacancia'}}}
idx=next((i for i,r in enumerate(registros) if r['id']==rid),None)
if idx is None: registros.append(registro)
else: registros[idx]=registro
print('Registro preparado:',rid)

## Vacância separada
A vacância não é misturada às vagas do edital. Informe a quantidade, a data de referência e o link da fonte específica.

In [ ]:
#@title 5. Atualizar vacância do cargo
ID_REGISTRO = "" #@param {type:"string"}
QUANTIDADE_VAGA = 0 #@param {type:"integer"}
DATA_REFERENCIA = "" #@param {type:"string"}
LINK_VACANCIA = "" #@param {type:"string"}
MOTIVO = "Quadro de cargos vagos consultado." #@param {type:"string"}
r=next((x for x in registros if x['id']==ID_REGISTRO),None)
if not r: print('Informe o ID exibido na célula anterior.')
else:
    r['vacancy']={'count':QUANTIDADE_VAGA,'reference_date':DATA_REFERENCIA,'reason':MOTIVO,'sources':[{'label':'Fonte da vacância','url':LINK_VACANCIA}] if LINK_VACANCIA else [],'collection_status':{'found':bool(LINK_VACANCIA),'code':'OK' if LINK_VACANCIA else 'VACANCY_NOT_FOUND','reason':'Vacância confirmada em fonte vinculada.' if LINK_VACANCIA else 'Quantidade informada sem link oficial.','source':'cadastro_colab'}}
    print('Vacância atualizada para',r['position'])

In [ ]:
#@title 6. Salvar, validar e gerar o portal
DATA_FILE.write_text(json.dumps(payload,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
subprocess.run(['python','src/build_site.py'],check=True)
subprocess.run(['python','src/test_examples.py'],check=True)
print('Portal atualizado em docs/index.html')

In [ ]:
#@title 7. Visualizar no Colab
from IPython.display import IFrame,display
import threading,http.server,socketserver
os.chdir(PASTA/'docs')
PORT=8000
try:
    httpd=socketserver.TCPServer(('',PORT),http.server.SimpleHTTPRequestHandler)
    threading.Thread(target=httpd.serve_forever,daemon=True).start()
except OSError: pass
display(IFrame(src=f'http://localhost:{PORT}',width='100%',height=720))

In [ ]:
#@title 8. Exportar ZIP pronto para o GitHub
from google.colab import files
os.chdir('/content')
saida=shutil.make_archive('/content/radar-concursos-ti-atualizado','zip',root_dir=PASTA)
print(saida)
files.download(saida)